In [5]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/appledisease/Test/Blotch_Apple/149.jpg
/kaggle/input/appledisease/Test/Blotch_Apple/143.jpg
/kaggle/input/appledisease/Test/Blotch_Apple/131.jpg
/kaggle/input/appledisease/Test/Blotch_Apple/130.jpg
/kaggle/input/appledisease/Test/Blotch_Apple/120.jpg
/kaggle/input/appledisease/Test/Blotch_Apple/124.jpg
/kaggle/input/appledisease/Test/Blotch_Apple/123.jpg
/kaggle/input/appledisease/Test/Blotch_Apple/137.jpg
/kaggle/input/appledisease/Test/Blotch_Apple/127.jpg
/kaggle/input/appledisease/Test/Blotch_Apple/140.jpg
/kaggle/input/appledisease/Test/Blotch_Apple/145.jpg
/kaggle/input/appledisease/Test/Blotch_Apple/135.jpg
/kaggle/input/appledisease/Test/Blotch_Apple/121.jpg
/kaggle/input/appledisease/Test/Blotch_Apple/126.jpg
/kaggle/input/appledisease/Test/Blotch_Apple/142.jpg
/kaggle/input/appledisease/Test/Blotch_Apple/148.jpg
/kaggle/input/appledisease/Test/Blotch_Apple/138.jpg
/kaggle/input/appledisease/Test/Blotch_Apple/146.jpg
/kaggle/input/appledisease/Test/Blotch_Apple/1

In [6]:
import os
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from PIL import Image
from sklearn.utils.class_weight import compute_class_weight
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import InceptionV3
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import GlobalAveragePooling2D, Dropout, Dense
from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping

In [7]:
train_dir = "/kaggle/input/appledisease/Train"
test_dir = "/kaggle/input/appledisease/Test"

In [8]:
img_size = (299, 299)  
batch_size = 32

train_datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2,
    rotation_range=25,
    zoom_range=0.3,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    horizontal_flip=True,
    brightness_range=[0.7, 1.3],
    fill_mode='nearest'
)

In [9]:
val_datagen = ImageDataGenerator(rescale=1./255, validation_split=0.2)
test_datagen = ImageDataGenerator(rescale=1./255)

In [10]:
train_gen = train_datagen.flow_from_directory(
    train_dir,
    target_size=img_size,
    batch_size=batch_size,
    class_mode='categorical',
    subset='training',
    shuffle=True
)

Found 307 images belonging to 4 classes.


In [11]:
val_gen = val_datagen.flow_from_directory(
    train_dir,
    target_size=img_size,
    batch_size=batch_size,
    class_mode='categorical',
    subset='validation',
    shuffle=False
)

Found 75 images belonging to 4 classes.


In [12]:
test_gen = test_datagen.flow_from_directory(
    test_dir,
    target_size=img_size,
    batch_size=batch_size,
    class_mode='categorical',
    shuffle=False
)

Found 120 images belonging to 4 classes.


In [13]:
for subdir, _, files in os.walk(train_dir):
    for file in files:
        try:
            img_path = os.path.join(subdir, file)
            img = Image.open(img_path)
            img.verify()
        except Exception as e:
            print(f"Corrupted or unreadable file: {img_path} — {e}")

In [14]:
labels_from_gen = train_gen.classes
class_weight = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(labels_from_gen),
    y=labels_from_gen
)
class_weight = dict(enumerate(class_weight))


In [15]:
lr_reduce = ReduceLROnPlateau(monitor='val_loss', patience=3, factor=0.5, verbose=1)
early_stop = EarlyStopping(monitor='val_loss', patience=7, restore_best_weights=True)

In [16]:
base_model = InceptionV3(include_top=False, weights='imagenet', input_shape=(299, 299, 3))
base_model.trainable = False

I0000 00:00:1764282942.642424      37 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 15513 MB memory:  -> device: 0, name: Tesla P100-PCIE-16GB, pci bus id: 0000:00:04.0, compute capability: 6.0


87910968/87910968 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step


In [17]:
model = Sequential()
model.add(base_model)
model.add(GlobalAveragePooling2D())
model.add(Dropout(0.3))
model.add(Dense(128, activation='relu'))
model.add(Dropout(0.3))
model.add(Dense(train_gen.num_classes, activation='softmax'))


In [18]:
model.compile(optimizer=tf.keras.optimizers.Adam(1e-4),
              loss='categorical_crossentropy',
              metrics=['accuracy'])


In [19]:
print('phase1...freez')
training = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=10,
    class_weight=class_weight,
    callbacks=[early_stop, lr_reduce]
)

phase1...freez


/usr/local/lib/python3.11/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/10


I0000 00:00:1764282967.586299     132 service.cc:148] XLA service 0x7ed8a8412750 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1764282967.587089     132 service.cc:156]   StreamExecutor device (0): Tesla P100-PCIE-16GB, Compute Capability 6.0
I0000 00:00:1764282969.546884     132 cuda_dnn.cc:529] Loaded cuDNN version 90300


 1/10 ━━━━━━━━━━━━━━━━━━━━ 3:21 22s/step - accuracy: 0.4062 - loss: 1.3507

I0000 00:00:1764282978.216474     132 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


10/10 ━━━━━━━━━━━━━━━━━━━━ 49s 3s/step - accuracy: 0.3616 - loss: 1.4130 - val_accuracy: 0.4800 - val_loss: 1.2308 - learning_rate: 1.0000e-04
Epoch 2/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 8s 826ms/step - accuracy: 0.4211 - loss: 1.3157 - val_accuracy: 0.6400 - val_loss: 1.1409 - learning_rate: 1.0000e-04
Epoch 3/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 8s 841ms/step - accuracy: 0.4666 - loss: 1.1744 - val_accuracy: 0.7333 - val_loss: 1.0447 - learning_rate: 1.0000e-04
Epoch 4/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 8s 842ms/step - accuracy: 0.5129 - loss: 1.1258 - val_accuracy: 0.7867 - val_loss: 0.9640 - learning_rate: 1.0000e-04
Epoch 5/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 8s 844ms/step - accuracy: 0.5730 - loss: 1.0441 - val_accuracy: 0.8133 - val_loss: 0.8912 - learning_rate: 1.0000e-04
Epoch 6/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 8s 832ms/step - accuracy: 0.6440 - loss: 0.9692 - val_accuracy: 0.8000 - val_loss: 0.8265 - learning_rate: 1.0000e-04
Epoch 7/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 8s 815ms/step - accuracy: 0.6276 - loss

In [20]:
base_model.trainable = True
model.compile(optimizer=tf.keras.optimizers.Adam(1e-5),
              loss='categorical_crossentropy',
              metrics=['accuracy'])

In [21]:
print("Phase...finetuning ")
finetuning = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=10,
    class_weight=class_weight,
    callbacks=[early_stop, lr_reduce]
)

Phase...finetuning 
Epoch 1/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 138s 5s/step - accuracy: 0.4954 - loss: 1.1556 - val_accuracy: 0.8533 - val_loss: 0.6146 - learning_rate: 1.0000e-05
Epoch 2/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 9s 861ms/step - accuracy: 0.6304 - loss: 0.9413 - val_accuracy: 0.8400 - val_loss: 0.5955 - learning_rate: 1.0000e-05
Epoch 3/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 9s 844ms/step - accuracy: 0.6879 - loss: 0.8622 - val_accuracy: 0.8133 - val_loss: 0.5718 - learning_rate: 1.0000e-05
Epoch 4/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 8s 809ms/step - accuracy: 0.7339 - loss: 0.7576 - val_accuracy: 0.8133 - val_loss: 0.5466 - learning_rate: 1.0000e-05
Epoch 5/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 9s 848ms/step - accuracy: 0.7113 - loss: 0.7928 - val_accuracy: 0.8267 - val_loss: 0.5280 - learning_rate: 1.0000e-05
Epoch 6/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 8s 821ms/step - accuracy: 0.8413 - loss: 0.5813 - val_accuracy: 0.8667 - val_loss: 0.4973 - learning_rate: 1.0000e-05
Epoch 7/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 9s 881m

In [22]:
best_train_acc = max(finetuning.history['accuracy'])
best_val_acc = max(finetuning.history['val_accuracy'])

In [23]:
print(f"Best Training Accuracy: {best_train_acc:.4f}")
print(f"Best Validation Accuracy: {best_val_acc:.4f}")

Best Training Accuracy: 0.8990
Best Validation Accuracy: 0.8933


In [24]:
print("Best Validation Accuracy:", max(finetuning.history['val_accuracy']))

Best Validation Accuracy: 0.8933333158493042
